# Awam Assist — Citizen Service Navigator
### RAG Pipeline Notebook

---

**Project Overview**

Awam Assist is a Retrieval-Augmented Generation (RAG) based chatbot designed to help Pakistani citizens navigate government services. It answers questions in plain English or Roman Urdu, retrieving answers from verified official government sources.

**Tech Stack**

| Component | Tool |
|-----------|------|
| Document Loading | LangChain TextLoader / PyPDFLoader |
| Text Splitting | RecursiveCharacterTextSplitter |
| Embeddings | HuggingFace `all-MiniLM-L6-v2` |
| Vector Store | ChromaDB |
| LLM | Groq — LLaMA 3.3 70B |
| Orchestration | LangChain LCEL |
| Deployment | FastAPI + Railway |

**Knowledge Base Categories**

1.  Zakat Punjab Programs
2.  IESCO Electricity Services
3.  Punjab Transport Services
4.  ICT Civil Registration (Marriage & Birth)
5.  NADRA Services
6.  BISP / Ehsaas Programs
7.  Rescue & Emergency Services
8.  Police & FIR Process
9.  Education and Scholarships
10. FBR Tax and NTN
11. Pakistan Citizens Portal
12. Passport Services
13. Property and Land Records
14. Sehat Sahulat Health
15. WASA and Gas SNGPL

---

## Step 1 — Install Dependencies

Install all required libraries. The `-q` flag suppresses verbose output.

> **Note:** After installation, restart the runtime before proceeding (`Runtime -> Restart session`).

In [3]:
!pip install langchain langchain-community langchain-groq langchain-text-splitters chromadb sentence-transformers pypdf -q

## Step 2 — Import Libraries

Import all necessary modules for the RAG pipeline:

- **Document Loaders** — read `.txt` and `.pdf` files from the knowledge base
- **Text Splitter** — chunk documents into smaller pieces for embedding
- **Embeddings** — convert text chunks into vector representations
- **Vector Store** — store and retrieve embeddings using ChromaDB
- **LLM** — Groq-hosted LLaMA 3.3 70B for answer generation
- **Chain Components** — LangChain LCEL for building the RAG pipeline

In [4]:
import os
import shutil
import zipfile

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All imports successful.")

All imports successful.


## Step 3 — Upload Knowledge Base

Upload your knowledge base zip file directly to Colab using the file picker below.

**Expected zip structure:**
```
RAG - Chatbot Citizen Service Navigator/
├── Zakat/
├── IESCO utility/
├── Transport services/
├── ICT marriage + birth certificate/
├── Nadra/
├── Ehsas and BISP Program/
├── Rescue and emergency/
├── Poilice and FIR/
├── Education and Scholarships/
├── FBR Tax and NTN/
├── Pakistan Citizens Portal/
├── Passport/
├── Property and Land Records/
├── Sehat Sahulat Health/
└── WASA and Gas SNGPL/
```

> PNG and image files are automatically skipped. Only `.txt` and `.pdf` files are loaded.

In [5]:
from google.colab import files as colab_files

print("Select your knowledge base zip file...")
uploaded = colab_files.upload()

# Automatically detect uploaded filename
ZIP_FILENAME = list(uploaded.keys())[0]
ZIP_PATH     = f"/content/{ZIP_FILENAME}"
EXTRACT_PATH = "/content/knowledge_base"
CHROMA_PATH  = "/content/chroma_db"

print(f"Uploaded: {ZIP_FILENAME}")

Select your knowledge base zip file...


Saving RAG - Chatbot Citizen Service Navigator (2).zip to RAG - Chatbot Citizen Service Navigator (2).zip
Uploaded: RAG - Chatbot Citizen Service Navigator (2).zip


## Step 4 — Extract and Load Documents

Extract the zip file and recursively load all `.txt` and `.pdf` files from every subfolder.

In [6]:
# Extract zip
if not os.path.exists(EXTRACT_PATH):
    print(f"Extracting {ZIP_FILENAME} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extraction complete.")
else:
    print("Already extracted, skipping.")

# Recursively load all .txt and .pdf files
docs         = []
skipped      = []

for root, dirs, files_list in os.walk(EXTRACT_PATH):
    for file in files_list:
        file_path = os.path.join(root, file)

        if file.endswith(".txt"):
            loader = TextLoader(file_path, encoding="utf-8")
            docs.extend(loader.load())
            print(f"  Loaded: {file}")

        elif file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            docs.extend(loader.load())
            print(f"  Loaded (PDF): {file}")

        else:
            skipped.append(file)

print(f"\nTotal documents loaded : {len(docs)}")
print(f"Files skipped          : {len(skipped)} (images/unsupported)")

Extracting RAG - Chatbot Citizen Service Navigator (2).zip ...
Extraction complete.
  Loaded: ICT Civil Registration Services.txt
  Loaded: Nadra complete.txt
  Loaded: WASA_Gas_SNGPL_Complete_Guide.txt
  Loaded: Property_Land_Records_Complete_Guide.txt
  Loaded: BISP_Ehsaas_Complete_Guide.txt
  Loaded: Emergency serice.txt
  Loaded: Iesco citizen guide.txt
  Loaded: Passport_Complete_Guide.txt
  Loaded: Police_FIR_Complete_Guide.txt
  Loaded: Sehat_Sahulat_Complete_Guide.txt
  Loaded: FBR_Tax_NTN_Complete_Guide.txt
  Loaded: Punjab transport complete.txt
  Loaded: Zakat punjab complete.txt
  Loaded: Education_Scholarships_Complete_Guide.txt
  Loaded: Pakistan_Citizens_Portal_Complete_Guide.txt

Total documents loaded : 15
Files skipped          : 0 (images/unsupported)


## Step 5 — Chunk Documents

Split documents into smaller chunks for effective retrieval.

**Parameters:**
- `chunk_size = 1000` — keeps section headers and their content together in one chunk
- `chunk_overlap = 100` — 100-character overlap prevents information loss at chunk boundaries

> chunk_size=1000 was chosen after testing showed that smaller sizes (500) caused section headers to split from their content, resulting in failed retrievals for some categories.

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print(f"Total chunks created : {len(chunks)}")
print(f"Avg chunk size       : ~{sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

Total chunks created : 110
Avg chunk size       : ~833 characters


## Step 6 — Clear Old ChromaDB

Delete any existing ChromaDB before building a fresh one. This prevents the read-only database error that occurs when an old index is already loaded in memory.

> Always run this cell before Step 7 when rebuilding the knowledge base.

In [8]:
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print("Old ChromaDB deleted.")
else:
    print("No existing ChromaDB found.")

No existing ChromaDB found.


## Step 7 — Generate Embeddings and Build Vector Store

Convert text chunks into numerical vector representations and store them in ChromaDB.

**Embedding Model:** `all-MiniLM-L6-v2`
- Lightweight 80MB model
- 384-dimensional dense vectors
- Runs locally with no API key required
- Strong performance on English text

> This step takes 1-2 minutes on first run as it downloads the embedding model.

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"Vector DB ready.")
print(f"Stored at    : {CHROMA_PATH}")
print(f"Total vectors: {vectordb._collection.count()}")

/tmp/ipykernel_5885/1237067122.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB ready.
Stored at    : /content/chroma_db
Total vectors: 110


## Step 8 — Initialize the LLM

Connect to Groq's API to use LLaMA 3.3 70B for answer generation.

**Why Groq?**
- Free tier available
- Extremely fast inference via LPU hardware
- LLaMA 3.3 70B is a high-quality open-source model

> Get your free API key at [console.groq.com](https://console.groq.com) and replace the placeholder below.

In [23]:
# ── Configuration ──────────────────────────────────────────────
GROQ_API_KEY = "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"   # Replace with your key
MODEL_NAME   = "llama-3.3-70b-versatile"
# ───────────────────────────────────────────────────────────────

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=MODEL_NAME
)

print(f"LLM ready: {MODEL_NAME}")

LLM ready: llama-3.1-8b-instant


## Step 9 — Build the RAG Chain

Assemble the full Retrieval-Augmented Generation pipeline using LangChain LCEL.

**Pipeline Flow:**
```
User Question
     |
     v
Retriever — searches ChromaDB for top-5 relevant chunks
     |
     v
Prompt Template — formats question + retrieved context
     |
     v
LLM (Groq LLaMA 3.3 70B) — generates answer from context only
     |
     v
Output Parser — returns clean string response
```

**Retriever k=5:** Fetches the top 5 most semantically similar chunks per query. Increased from 3 after testing showed some answers existed in chunks ranked 4th or 5th for categories like Passport and Civil Registration.

In [24]:
SYSTEM_PROMPT = """
You are a helpful citizen service assistant for Pakistan.
Answer the question based only on the context provided below.
Use simple, plain language that any citizen can understand.
If the user writes in Roman Urdu, reply in Roman Urdu.
If the user writes in English, reply in English.
If the answer is not in the context, say: "Mujhe is baray mein maloomat nahi. Please relevant department se rabta karein."
Keep answers concise and practical.

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)

retriever = vectordb.as_retriever(search_kwargs={"k": 5})

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


## Step 10 — Single Query Test

Run a single test query to verify the full pipeline is working end-to-end.

In [25]:
query = "Am I eligible for Zakat Guzara allowance?"

print(f"Question: {query}")
print(f"\nAnswer:")
print(chain.invoke(query))

Question: Am I eligible for Zakat Guzara allowance?

Answer:
Apko Zakat Guzara allowance ke liye eligible hone ke liye kuch sharton ka poora hona zaroori hai:

- Aapko mustakil Muslim hona chahiye.
- Aapko ghar ki arthvyavastha mein kami honi chahiye.
- Aapko koi nishchit kaam nahi hai.
- Aapki istehqaq ka faisla local Zakat aur ushr committee dwara kiya jaega.

Agar aapko in sabhi sharton ka poora hona hai to aap Zakat Guzara allowance ke liye eligible ho sakte hain.


## Step 11 — Batch Evaluation Across All 15 Categories

Test the chatbot with one question per knowledge base category to verify full coverage and response quality.

In [26]:
test_questions = [
    ("Zakat",           "What is the monthly Guzara allowance amount from Zakat?"),
    ("IESCO",           "How do I apply for a new electricity connection from IESCO?"),
    ("Transport",       "What is the T-Cash card and how do I get one?"),
    ("Civil Reg",       "What documents are needed to register a marriage in Islamabad?"),
    ("NADRA",           "How do I renew my CNIC and how much does it cost?"),
    ("BISP",            "How do I check if I am eligible for BISP Kafaalat program?"),
    ("Emergency",       "What is the emergency helpline number in Punjab?"),
    ("Police/FIR",      "What should I do if police refuse to register my FIR?"),
    ("Education",       "How do I apply for a government scholarship in Pakistan?"),
    ("FBR / Tax",       "How do I register for NTN with FBR?"),
    ("Citizens Portal", "How do I file a complaint on the Pakistan Citizens Portal?"),
    ("Passport",        "What documents are needed to apply for a Pakistani passport?"),
    ("Property",        "How do I check my land records in Punjab?"),
    ("Sehat Sahulat",   "Who is eligible for the Sehat Sahulat Health card?"),
    ("WASA / Gas",      "How do I apply for a new SNGPL gas connection?"),
]

print("=" * 70)
print("  AWAM ASSIST — BATCH EVALUATION")
print("=" * 70)

for category, question in test_questions:
    print(f"\nCategory : {category}")
    print(f"Question : {question}")
    print(f"Answer   : {chain.invoke(question)}")
    print("-" * 70)

  AWAM ASSIST — BATCH EVALUATION

Category : Zakat
Question : What is the monthly Guzara allowance amount from Zakat?
Answer   : The monthly Guzara allowance amount from Zakat for chronic poor people is PKR 2,000.
----------------------------------------------------------------------

Category : IESCO
Question : How do I apply for a new electricity connection from IESCO?
Answer   : Aap IESCO ke liye nayi electricity connection ke liye apply karne ke liye apne sabse kareeb IESCO ke sub-division office mein ja sakte hain. Iske alawa, aap telephone, mail ya IESCO ki website ke madhyam se bhi apply kar sakte hain. Har sub-division mein ek one-window operation facility bhi hai.
----------------------------------------------------------------------

Category : Transport
Question : What is the T-Cash card and how do I get one?
Answer   : T-Cash Card hai apni CNIC ke sath online apply karein ya app download karein. Apne CNIC aur mobile number jameen karke Rs. 130 ki fee payein aur apna card on

## Step 12 — Debug: Inspect Retrieved Chunks

Use this cell to debug any category that returns an incorrect or empty answer. It shows exactly which chunks are being retrieved and from which source file.

In [27]:
# Change this query to debug any failing category
debug_query = "What documents are needed to apply for a Pakistani passport?"

retrieved = retriever.invoke(debug_query)

print(f"Query: {debug_query}")
print("=" * 60)

for i, chunk in enumerate(retrieved):
    source = chunk.metadata.get('source', 'unknown').split('/')[-1]
    print(f"\nChunk {i+1} — Source: {source}")
    print("-" * 40)
    print(chunk.page_content[:300])
    print("=" * 60)

Query: What documents are needed to apply for a Pakistani passport?

Chunk 1 — Source: Passport_Complete_Guide.txt
----------------------------------------
PASSPORT SERVICES - COMPLETE CITIZEN GUIDE
Directorate General of Immigration & Passports (DGIP)
Applicable: All Pakistani Citizens | Updated: 2025

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SECTION 1: TYPES OF PASSPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



Chunk 2 — Source: Nadra complete.txt
----------------------------------------
Online portal: onlinemrp.dgip.gov.pk
Fee payment: Passport Fee Asaan App or e-Payment Web Portal

Types:
- Ordinary Machine Readable Passport (MRP)
- e-Passport (available from all field offices since August 2023)

Documents required:
- Original valid CNIC/NICOP with photocopy
- Previous passport wi

Chunk 3 — Source: Nadra complete.txt
----------------------------------------
NICOP (NATIONAL IDENTITY CARD FOR OVERSEAS PAKISTANIS)
For Pakistani citizens living abroad. Allows dual nationality holders to

## Step 13 — Verify Indexed Files

Confirm all 15 knowledge base files are in ChromaDB and check total vector count.

In [28]:
print(f"Total vectors in DB: {vectordb._collection.count()}")

results = vectordb._collection.get(include=["metadatas"])
sources = set(m.get('source', '').split('/')[-1] for m in results['metadatas'])

print(f"\nFiles indexed ({len(sources)}):")
for s in sorted(sources):
    print(f"  {s}")

Total vectors in DB: 110

Files indexed (15):
  BISP_Ehsaas_Complete_Guide.txt
  Education_Scholarships_Complete_Guide.txt
  Emergency serice.txt
  FBR_Tax_NTN_Complete_Guide.txt
  ICT Civil Registration Services.txt
  Iesco citizen guide.txt
  Nadra complete.txt
  Pakistan_Citizens_Portal_Complete_Guide.txt
  Passport_Complete_Guide.txt
  Police_FIR_Complete_Guide.txt
  Property_Land_Records_Complete_Guide.txt
  Punjab transport complete.txt
  Sehat_Sahulat_Complete_Guide.txt
  WASA_Gas_SNGPL_Complete_Guide.txt
  Zakat punjab complete.txt


## Step 14 — Interactive Chat

Test the chatbot interactively inside the notebook.

In [29]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_input = widgets.Text(
    placeholder="Ask about any Pakistani government service...",
    description="Question:",
    layout=widgets.Layout(width="650px")
)

output_area = widgets.Output()

ask_button = widgets.Button(
    description="Ask",
    button_style="primary",
    layout=widgets.Layout(width="100px")
)

def on_ask(b):
    with output_area:
        clear_output()
        if text_input.value.strip():
            print("Thinking...")
            answer = chain.invoke(text_input.value)
            clear_output()
            print(f"Q: {text_input.value}")
            print(f"\nA: {answer}")
        else:
            print("Please enter a question.")

ask_button.on_click(on_ask)

print("Awam Assist — Interactive Chat")
print("Covers: Zakat, NADRA, BISP, IESCO, Transport, Emergency, Marriage/Birth, FIR, Education, FBR, Passport, Property, Sehat Sahulat, WASA/Gas, Citizens Portal")
display(text_input, ask_button, output_area)

Awam Assist — Interactive Chat
Covers: Zakat, NADRA, BISP, IESCO, Transport, Emergency, Marriage/Birth, FIR, Education, FBR, Passport, Property, Sehat Sahulat, WASA/Gas, Citizens Portal


Text(value='', description='Question:', layout=Layout(width='650px'), placeholder='Ask about any Pakistani gov…

Button(button_style='primary', description='Ask', layout=Layout(width='100px'), style=ButtonStyle())

Output()

## Step 15 — Export ChromaDB for Deployment

Download the ChromaDB vector store to deploy with the FastAPI backend on Railway.

After downloading, extract the zip and replace these 5 files in your Railway GitHub repo:
```
chroma.sqlite3
data_level0.bin
header.bin
length.bin
link_lists.bin
```
Push to GitHub and Railway will auto-redeploy with the updated knowledge base.

In [31]:
from google.colab import files as colab_files

OUTPUT_ZIP = "/content/chroma_db_exports"

print("Zipping ChromaDB...")
shutil.make_archive(OUTPUT_ZIP, 'zip', CHROMA_PATH)
print(f"Saved to {OUTPUT_ZIP}.zip")

print("Starting download...")
colab_files.download(f"{OUTPUT_ZIP}.zip")

Zipping ChromaDB...
Saved to /content/chroma_db_exports.zip
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Architecture Reference

```
Raw Documents (.txt / .pdf)
        |
        v
TextLoader / PyPDFLoader
        |
        v
RecursiveCharacterTextSplitter  (chunk_size=1000, overlap=100)
        |
        v
HuggingFace Embeddings  (all-MiniLM-L6-v2, 384 dimensions)
        |
        v
ChromaDB  (persistent vector store)
        |
   At query time
        |
        v
Semantic Search  (top-k=5 chunks by cosine similarity)
        |
        v
Prompt Template + LangChain LCEL
        |
        v
Groq LLaMA 3.3 70B
        |
        v
Plain-language answer in English or Roman Urdu
```

## Links

- **Live API:** https://awam-assist-production.up.railway.app
- **Frontend:** https://awamassist.vercel.app
- **GitHub:** https://github.com/MurtazaMajid/Awam-Asist-Citizen-Services-Navigation-RAG-Chatbot
- **Groq Console:** https://console.groq.com

---
*Awam Assist — Murtaza Majid, 2026*